# Diagnostic des données extraites

Avant d'écrire le pipeline de transformation, je regarde les données produites par l'extraction (étape 2) pour repérer les erreurs à corriger. Je reprends les types d'erreurs du cours : valeurs manquantes, erreurs de formatage, doublons, valeurs extrêmes.

Ce notebook sert seulement à explorer. Les corrections sont faites dans `src/transform.py`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from PIL import Image

ROOT = Path("..")
files = sorted((ROOT / "data" / "extracted").glob("articles_*.json"))

dataframes = []
for file in files:
    with open(file, encoding="utf-8") as f:
        df_file = pd.DataFrame(json.load(f))
    df_file["fichier"] = file.name
    dataframes.append(df_file)

df = pd.concat(dataframes, ignore_index=True)
print(len(files), "fichiers,", df.shape[0], "articles,", df.shape[1], "colonnes")

2 fichiers, 251 articles, 14 colonnes


In [2]:
df.head(3)

,id,source,domaine,url,titre,texte,image_url,image_path,date_publication,langue,auteur,fiabilite_source,label,fichier
0,bd7ce5a86e742d2c77a2e1d57a26eb7a,newsdata,france24.com,https://www.france24.com/fr/%C3%A9co-tech/2026...,Nouveau test pour SpaceX : sa mégafusée Starsh...,L'entreprise d'Elon Musk doit tenter lundi d'a...,https://s.france24.com/media/display/3c5e7be4-...,data/images/bd7ce5a86e742d2c77a2e1d57a26eb7a.jpg,2026-09-27T22:46:11+00:00,fr,france 24,3118.0,NaN,articles_20260928_110726.json
1,d0f543bcb715efe80244bbb2244fc6b8,newsdata,laprovence.com,https://www.laprovence.com/article/france-mond...,SpaceX vise pour la première fois l'orbite ter...,Après son entrée retentissante en Bourse au mo...,https://pictures.laprovence.com/media/afp/a53e...,data/images/d0f543bcb715efe80244bbb2244fc6b8.jpg,2026-09-27T22:43:31+00:00,fr,afp import,6688.0,NaN,articles_20260928_110726.json
2,b09b49a8fb602543060c84a7c9b8e23d,newsdata,epochtimes.fr,https://www.epochtimes.fr/italie-un-rappeur-mo...,Italie : un rappeur moldave expulsé après avoi...,"Connu sous le nom de scène Bratan, Nicolae Nov...",https://images-fr.epochtimes.fr/uploads/2026/0...,data/images/b09b49a8fb602543060c84a7c9b8e23d.jpg,2026-09-27T22:37:00+00:00,fr,epochtimes.fr avec afp,10731.0,NaN,articles_20260928_110726.json


## 1. Valeurs manquantes

In [3]:
df.isnull().sum()

id                    0
source                0
domaine               0
url                   0
titre                 0
texte                 0
image_url             0
image_path            0
date_publication      0
langue                0
auteur              127
fiabilite_source    187
label               231
fichier               0
dtype: int64

In [4]:
# Répartition des valeurs manquantes par source
df.groupby("source")[["auteur", "fiabilite_source", "label"]].apply(lambda g: g.isnull().sum())

,auteur,fiabilite_source,label
source,,,
20minutes_fakeoff,0,40,40
franceinfo,55,55,55
legorafi,0,20,0
lemonde,32,32,32
lesdecodeurs,40,40,40
newsdata,0,0,64


- `auteur` manque souvent (les flux RSS ne le donnent pas toujours) : on le remplacera par "inconnu"
- `fiabilite_source` n'existe que pour NewsData : on le laisse vide pour les autres sources, le remplir serait inventer une valeur
- `label` n'existe que pour Le Gorafi ("satire") : pour les autres, personne n'a vérifié l'article, on mettra "non_verifie"

## 2. Types

In [5]:
df.dtypes

id                      str
source                  str
domaine                 str
url                     str
titre                   str
texte                   str
image_url               str
image_path              str
date_publication        str
langue                  str
auteur                  str
fiabilite_source    float64
label                   str
fichier                 str
dtype: object

In [6]:
print(df["date_publication"].head(3).tolist())
print(df["fiabilite_source"].dropna().head(3).tolist())

['2026-09-27T22:46:11+00:00', '2026-09-27T22:43:31+00:00', '2026-09-27T22:37:00+00:00']
[3118.0, 6688.0, 10731.0]


Les dates sont du texte et le rang de fiabilité est un décimal (`3118.0`) à cause des valeurs manquantes. Il faudra convertir les dates en datetime et le rang en entier.

## 3. Formatage des URL et doublons

In [7]:
# URL avec des paramètres après ? ou #
df["url"].str.contains(r"[?#]").groupby(df["source"]).sum()

source
20minutes_fakeoff    40
franceinfo           55
legorafi              0
lemonde               0
lesdecodeurs          0
newsdata              0
Name: url, dtype: int64

In [8]:
print(df.loc[df["source"] == "franceinfo", "url"].iloc[0])
print(df.loc[df["source"] == "20minutes_fakeoff", "url"].iloc[0])

https://www.franceinfo.fr/politique/rachida-dati/direct-proces-de-rachida-dati-suivez-le-dernier-jour-d-audience-et-les-requisitions-du-parquet-national-financier_8209622.html#xtor=RSS-3-[lestitres]
https://www.20minutes.fr/societe/4248812-20260925-prix-litteraire-peut-etre-decerne-livre-ecrit-aide-ia?at_medium=display&at_campaign=149


In [9]:
print("Doublons sur l'URL :", df.duplicated(subset="url").sum())
url_sans_suivi = df["url"].str.split("#").str[0].str.split("?").str[0]
print("Doublons sur l'URL sans paramètres :", url_sans_suivi.duplicated().sum())

Doublons sur l'URL : 114
Doublons sur l'URL sans paramètres : 114


- le même article revient à chaque extraction : il faut supprimer les doublons
- les URL de franceinfo et de 20 Minutes contiennent des paramètres de suivi (`#xtor=`, `?at_medium=`). Pour l'instant ils sont identiques d'une extraction à l'autre, donc ils ne changent pas le nombre de doublons. On les retire quand même : si un paramètre change (une autre campagne par exemple), le même article aurait deux URL différentes

## 4. Textes

In [10]:
df["texte"].str.len().describe()

count     251.000000
mean      312.039841
std       359.907286
min        77.000000
25%       159.000000
50%       209.000000
75%       258.500000
max      1642.000000
Name: texte, dtype: float64

In [11]:
fin_coupee = df["texte"].str.contains(r"(?:\.\.\.|…)$")
print("Textes qui finissent par ... :", fin_coupee.sum())
print(df.loc[fin_coupee, "texte"].iloc[0][-80:])

Textes qui finissent par ... : 11
jectifs ambitieux.Ce test, devant durer environ dix heures, doit voir l'étage...


In [12]:
print("Titres avec apostrophe typographique ’ :", df["titre"].str.contains("’").sum())
print("Titres avec apostrophe simple ' :", df["titre"].str.contains("'").sum())

Titres avec apostrophe typographique ’ : 87
Titres avec apostrophe simple ' : 56


- aucun texte vide ou très court
- certaines descriptions NewsData sont coupées et finissent par "..." : on retire ces points de suspension
- les deux types d'apostrophes sont mélangés (erreur d'irrégularité) : on garde seulement l'apostrophe simple

## 5. Images et association texte-image

In [13]:
# Même image pour plusieurs articles différents (après retrait des doublons)
sans_doublons = df[~url_sans_suivi.duplicated()]
partagees = sans_doublons[sans_doublons["image_url"].duplicated(keep=False)]
partagees[["domaine", "titre", "image_url"]]

,domaine,titre,image_url
5,lequipe.fr,Le ministre israélien des Affaires étrangères ...,https://medias.lequipe.fr/img-photo-png/-/1500...
8,lequipe.fr,"Avec 38 points, A'ja Wilson porte Las Vegas fa...",https://medias.lequipe.fr/img-photo-png/-/1500...
10,lequipe.fr,Paul Seixas doit-il s'en vouloir ?,https://medias.lequipe.fr/img-photo-png/-/1500...
18,lequipe.fr,"Un combat de rue, du suspense et la surprise M...",https://medias.lequipe.fr/img-photo-png/-/1500...
35,lemonde.fr,"EN DIRECT, guerre en Ukraine : les attaques ru...",https://img.lemde.fr/2026/09/28/845/0/6000/300...
162,lemonde.fr,"EN DIRECT, guerre en Ukraine : les attaques ru...",https://img.lemde.fr/2026/09/28/845/0/6000/300...


In [14]:
# Nom du fichier image = id de l'article ?
nom_ok = df["image_path"].apply(lambda p: Path(p).stem) == df["id"]
print("Images dont le nom ne correspond pas à l'id :", (~nom_ok).sum())

Images dont le nom ne correspond pas à l'id : 0


In [15]:
tailles = []
for path in df["image_path"].unique():
    with Image.open(ROOT / path) as img:
        tailles.append({"largeur": img.width, "hauteur": img.height, "format": img.format})
tailles = pd.DataFrame(tailles)
print(tailles["format"].value_counts())
tailles.describe()

format
JPEG    124
PNG      13
Name: count, dtype: int64


,largeur,hauteur
count,137.000000,137.000000
mean,775.437956,459.160584
std,322.943331,226.217400
min,107.000000,56.000000
25%,432.000000,243.000000
50%,644.000000,322.000000
75%,1024.000000,600.000000
max,2000.000000,1333.000000


In [16]:
tailles.sort_values("largeur").head(3)

,largeur,hauteur,format
126,107,56,PNG
22,300,225,JPEG
17,300,225,JPEG


- toutes les images correspondent à l'id de leur article
- certaines images sont utilisées par plusieurs articles différents. Chez L'Équipe, c'est une image par défaut : elle n'illustre pas l'article, on écarte ces articles. Chez Le Monde, c'est un direct dont l'URL a changé avec le titre
- quelques images sont très petites (valeurs extrêmes) : on fixe une largeur minimale de 200 px

## 6. Langue et dates

In [17]:
print(df["langue"].value_counts())
dates = pd.to_datetime(df["date_publication"], utc=True)
print("Plus ancienne :", dates.min())
print("Plus récente :", dates.max())

langue
fr    251
Name: count, dtype: int64
Plus ancienne : 2026-09-16 14:00:00+00:00
Plus récente : 2026-09-28 11:23:16+00:00


Une seule langue et pas de date aberrante pour l'instant. Le pipeline vérifiera quand même ces deux points, car les prochaines extractions pourraient en contenir.

## Conclusion

Traitements à faire dans le pipeline (`src/transform.py`) :

- retirer les paramètres de suivi des URL : `nettoie_url()`
- supprimer les doublons sur l'URL nettoyée : `supprime_doublons()`
- uniformiser les apostrophes et retirer les "..." de fin : `nettoie_texte()`
- convertir les dates et le rang de fiabilité : `convertit_types()`
- écarter les images trop petites : `valide_image()`
- écarter les images partagées entre plusieurs articles : `verifie_association()`
- remplacer l'auteur et le label manquants : `remplit_manquants()`